# Lab 8: Stabilized FEM for the Stokes problem

On the unit square domain $\Omega$ we consider the Stokes problem expressed by 
\begin{align*} 
- \boldsymbol{\Delta}\boldsymbol{u} +\boldsymbol{\nabla}  p  &= \boldsymbol{f} \quad {\rm in} \ \Omega, \\ 
\boldsymbol{\nabla} \cdot \boldsymbol{u} &= 0 \quad {\rm in} \ \Omega, \\ 
\boldsymbol{u} &= \boldsymbol{u}_D \;\; {\rm on} \ \Gamma_D, \\
(\boldsymbol{\nabla} \boldsymbol{u} - p)\cdot\boldsymbol{n} &= \boldsymbol{g}_N\;\; {\rm on}\ \partial\Omega \setminus \Gamma_D.
\end{align*}

## Saddle-point weak formulation

The weak formulation is derived by testing the first equation with a vector-valued function $\boldsymbol{v}$ in $H^1_0(\Omega)^d$ and the second equation with $q\in L^2(\Omega)$. Assuming, for the sake of simplicity, that $\boldsymbol{u}_D = \boldsymbol{0}$, and integrating by parts in the first equation, we end up with the saddle point problem
\begin{align*}
\int_{\Omega} \boldsymbol{\nabla} \boldsymbol{u} \cdot \boldsymbol{\nabla} \boldsymbol{v}
- (\boldsymbol{\nabla} \cdot \boldsymbol{v}) \ p \ {\rm d} x
&= \int_{\Omega} \boldsymbol{f} \cdot \boldsymbol{v} \, {\rm d} x + \int_{\Gamma_N} \boldsymbol{g} \cdot \boldsymbol{v} \, {\rm d} s. \\
\int_\Omega (\boldsymbol{\nabla} \cdot \boldsymbol{u}) \ q \ {\rm d} x &= 0
\end{align*}

Defining $a: H^1_0(\Omega)^d\times H^1_0(\Omega)^d$ and $b:H^1_0(\Omega)^d\times L^2(\Omega)$ such that
\begin{align*}
a(\boldsymbol{v},\boldsymbol{w}) &:=\int_{\Omega} \boldsymbol{\nabla} \boldsymbol{v} \cdot \boldsymbol{\nabla} \boldsymbol{w}\  {\rm d} x, 
\quad\forall\ v,w\in H^1_0(\Omega)^d \\ 
b(\boldsymbol{v},q) &:= -\int_\Omega (\boldsymbol{\nabla} \cdot \boldsymbol{v}) \ q \ {\rm d} x, \quad\forall\ v\in H^1_0(\Omega)^d \;\text{and}\; \forall\ q\in L^2(\Omega),
\end{align*}
the weak form of Stokes problem reads: Find $(u,p)$ such that
\begin{align*}
a(\boldsymbol{u},\boldsymbol{v}) + b(\boldsymbol{v},p)
&= \int_{\Omega} \boldsymbol{f} \cdot \boldsymbol{v} \, {\rm d} x + \int_{\Gamma_N} \boldsymbol{g} \cdot \boldsymbol{v} \, {\rm d} s. \\
b(\boldsymbol{u},q) &= 0.
\end{align*}

## Numerical solution with FEniCS

In [1]:
%%capture
!apt-get install -y -qq software-properties-common python-software-properties module-init-tools
!add-apt-repository -y ppa:fenics-packages/fenics
!apt-get update -qq
!apt install -y --no-install-recommends fenics
!rm -rf *
from fenics import *

We want to compare the **stabilized $\mathbb{P}^1$-$\mathbb{P}^1$ FEM** for Stokes with respect to two other first order stable elements, i.e. the **MINI element** and the **Crouzeix--Raviart element**.

#### FE solution with MINI element

In [2]:
def mini_stokes(n, u_exact, p_exact, f, gN):    
    # 1. generate the mesh and mark the boundaries
    mesh = UnitSquareMesh(n, n, 'crossed')
    
    boundary_markers = MeshFunction('size_t', mesh, mesh.geometric_dimension()-1, 0)

    class top_boundary(SubDomain):
        def inside(self, x, on_boundary):
            return on_boundary and near(x[1], 1)

    class other_boundary(SubDomain):
        def inside(self, x, on_boundary):
            return on_boundary and not near(x[1], 1)

    top = top_boundary()
    top.mark(boundary_markers, 1)
    other = other_boundary()
    other.mark(boundary_markers, 2)
    
    ds = Measure('ds', domain=mesh, subdomain_data=boundary_markers)

    # 2. finite element spaces and Dirichlet BC
    Q = FiniteElement('CG', mesh.ufl_cell(), 1)
    B = FiniteElement("Bubble", mesh.ufl_cell(), mesh.topology().dim() + 1)
    V = VectorElement(NodalEnrichedElement(Q, B))
    X = FunctionSpace(mesh, V * Q)

    bc = DirichletBC(X.sub(0), u_exact, boundary_markers, 2)

    # 3. problem definition
    u, p = TrialFunctions(X)
    v, q = TestFunctions(X)

    a = (inner(grad(u), grad(v)) - p*div(v) - div(u)*q) * dx
    L = dot(f, v) * dx + dot(gN, v) * ds(1)

    # 4. solution
    x = Function(X)
    solve(a == L, x, bc)

    u, p = x.split()
    return u, p

#### FE solution with Crouzeix-Raviart element

In [3]:
def cr_stokes(n, u_exact, p_exact, f, gN):    
    # 1. generate the mesh and mark the boundaries
    mesh = UnitSquareMesh(n, n, 'crossed')
    
    boundary_markers = MeshFunction('size_t', mesh, mesh.geometric_dimension()-1, 0)

    class top_boundary(SubDomain):
        def inside(self, x, on_boundary):
            return on_boundary and near(x[1], 1)

    class other_boundary(SubDomain):
        def inside(self, x, on_boundary):
            return on_boundary and not near(x[1], 1)

    top = top_boundary()
    top.mark(boundary_markers, 1)
    other = other_boundary()
    other.mark(boundary_markers, 2)
    
    ds = Measure('ds', domain=mesh, subdomain_data=boundary_markers)

    # 2. finite element space and Dirichlet BC
    V = VectorElement('CR', mesh.ufl_cell(), 1)
    Q = FiniteElement('DG', mesh.ufl_cell(), 0)
    X = FunctionSpace(mesh, V * Q)

    bc = DirichletBC(X.sub(0), u_exact, boundary_markers, 2)

    # 3. problem definition
    u, p = TrialFunctions(X)
    v, q = TestFunctions(X)

    a = (inner(grad(u), grad(v)) - p*div(v) - div(u)*q) * dx
    L = dot(f, v) * dx + dot(gN, v) * ds(1)

    # 4. solution
    x = Function(X)
    solve(a == L, x, bc)

    u, p = x.split()
    return u, p

#### FE solution with SUPG stabilized P1-P1 element

In [4]:
def supg_stokes(n, u_exact, p_exact, f, gN):    
    # 1. generate the mesh and mark the boundaries
    mesh = UnitSquareMesh(n, n, 'crossed')
    
    boundary_markers = MeshFunction('size_t', mesh, mesh.geometric_dimension()-1, 0)

    class top_boundary(SubDomain):
        def inside(self, x, on_boundary):
            return on_boundary and near(x[1], 1)

    class other_boundary(SubDomain):
        def inside(self, x, on_boundary):
            return on_boundary and not near(x[1], 1)

    top = top_boundary()
    top.mark(boundary_markers, 1)
    other = other_boundary()
    other.mark(boundary_markers, 2)
    
    ds = Measure('ds', domain=mesh, subdomain_data=boundary_markers)

    # 2. finite element space and Dirichlet BC
    V = VectorElement('CG', mesh.ufl_cell(), 1)
    Q = FiniteElement('CG', mesh.ufl_cell(), 1)
    X = FunctionSpace(mesh, V * Q)

    bc = DirichletBC(X.sub(0), u_exact, boundary_markers, 2)

    # 3. problem definition
    u, p = TrialFunctions(X)
    v, q = TestFunctions(X)

    a = (inner(grad(u), grad(v)) - p*div(v) - div(u)*q) * dx
    L = dot(f, v) * dx + dot(gN, v) * ds(1)
  
    h = CellDiameter(X.mesh())
    tau_K = 0.5 * (h**2) 
    a += tau_K * inner(grad(p), grad(q)) * dx
    L += tau_K * dot(f,grad(q)) * dx  

    # 4. solution
    x = Function(X)
    solve(a == L, x, bc)

    u, p = x.split()
    return u, p

#### FE solution with P1-P1 element stabilized with pressure mass matrix

In [5]:
def massstab_stokes(n, u_exact, p_exact, f, gN):    
    # 1. generate the mesh and mark the boundaries
    mesh = UnitSquareMesh(n, n, 'crossed')
    
    boundary_markers = MeshFunction('size_t', mesh, mesh.geometric_dimension()-1, 0)

    class top_boundary(SubDomain):
        def inside(self, x, on_boundary):
            return on_boundary and near(x[1], 1)

    class other_boundary(SubDomain):
        def inside(self, x, on_boundary):
            return on_boundary and not near(x[1], 1)

    top = top_boundary()
    top.mark(boundary_markers, 1)
    other = other_boundary()
    other.mark(boundary_markers, 2)
    
    ds = Measure('ds', domain=mesh, subdomain_data=boundary_markers)

    # 2. finite element space and Dirichlet BC
    V = VectorElement('CG', mesh.ufl_cell(), 1)
    Q = FiniteElement('DG', mesh.ufl_cell(), 0)
    X = FunctionSpace(mesh, V * Q)

    bc = DirichletBC(X.sub(0), u_exact, boundary_markers, 2)

    # 3. problem definition
    u, p = TrialFunctions(X)
    v, q = TestFunctions(X)

    a = (inner(grad(u), grad(v)) - p*div(v) - div(u)*q) * dx
    L = dot(f, v) * dx + dot(gN, v) * ds(1)

    h = CellDiameter(mesh)
    a_stab = a + (h**2) * p * q * dx

    # 4. solution
    x = Function(X)
    solve(a_stab == L, x, bc)

    u, p = x.split()
    return u, p

### Convergence analysis and comparizon of the different methods

Compute the functions $\boldsymbol{f}$, $\boldsymbol{g}_N$ and $\boldsymbol{u}_D$ corresponding to the analytical solution
$$
            \boldsymbol{u}(x,y) = \begin{bmatrix}-\cos(\pi x)\sin(\pi y) \\ \sin(\pi x)\cos(\pi y)\end{bmatrix},
            \qquad
            p(x,y) = -\frac{1}{4}(\cos{(2\pi x)}+\cos{(2\pi y)}).
$$

Assess the convergence, measuring the errors as 
$$
||\boldsymbol{u} - \boldsymbol{u}_h||_{H^1(\Omega)}, \quad\text{ and}\quad ||p - p_h||_ {L^2(\Omega)}.
$$

In [7]:
u_exact = Expression((
        '-cos(pi*x[0]) * sin(pi*x[1])',
        'sin(pi*x[0]) * cos(pi*x[1])'
                    ), degree=2)
p_exact = Expression(
        '-0.25 * (cos(2*pi*x[0]) + cos(2*pi*x[1]))',
        degree=2)
f = Expression((
        '-2*pi*pi * cos(pi*x[0]) * sin(pi*x[1]) + 0.5*pi * sin(2*pi*x[0])',
        '2*pi*pi * sin(pi*x[0]) * cos(pi*x[1]) + 0.5*pi * sin(2*pi*x[1])'
               ), degree=2)

gN = Expression(('pi * cos(pi*x[0])', '0.25 * (cos(2*pi*x[0]) + cos(2*pi*x[1]))'
                ), degree=2)

for n in [10]: #, 20, 40, 80]:
    uhb, phb = mini_stokes(n, u_exact, p_exact, f, gN)
    
    eL2 = errornorm(p_exact, phb, 'L2')
    eH1 = errornorm(u_exact, uhb, 'H1')

    print('n={} eL2={:.2e} eH1={:.2e}'.format(n, eL2, eH1))

print()
for n in [10, 20, 40, 80]:
    uhcr, phcr = cr_stokes(n, u_exact, p_exact, f, gN)
    
    eL2 = errornorm(p_exact, phcr, 'L2')
    eH1 = errornorm(u_exact, uhcr, 'H1')

    print('n={} eL2={:.2e} eH1={:.2e}'.format(n, eL2, eH1))

print()
for n in [10]:
    uhsuph, phsuph = supg_stokes(n, u_exact, p_exact, f, gN)
    
    eL2 = errornorm(p_exact, phsuph, 'L2')
    eH1 = errornorm(u_exact, uhsuph, 'H1')

    print('n={} eL2={:.2e} eH1={:.2e}'.format(n, eL2, eH1))

print()
for n in [10, 20, 40, 80]:
    uh, ph = massstab_stokes(n, u_exact, p_exact, f, gN)
    
    eL2 = errornorm(p_exact, ph, 'L2')
    eH1 = errornorm(u_exact, uh, 'H1')

    print('n={} eL2={:.2e} eH1={:.2e}'.format(n, eL2, eH1))

n=10 eL2=2.16e-01 eH1=3.03e-01

n=10 eL2=3.88e-01 eH1=4.75e-01
n=20 eL2=1.56e-01 eH1=2.47e-01
n=40 eL2=7.03e-02 eH1=1.25e-01
n=80 eL2=3.41e-02 eH1=6.28e-02

n=10 eL2=2.01e+01 eH1=1.88e+01

n=10 eL2=4.37e-01 eH1=3.32e-01
n=20 eL2=2.21e-01 eH1=1.66e-01
n=40 eL2=1.13e-01 eH1=8.31e-02
n=80 eL2=5.76e-02 eH1=4.15e-02


In [ ]:
plot(ph)

In [ ]:
plot(uhb)